# CNN Model

### Import packages and load data

In [126]:
%reload_ext autoreload
%autoreload 2
%aimport -numpy, -pandas, -matplotlib

import sys
import yaml
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
import matplotlib.pyplot as plt
# use same font as latex
# plt.rc('text', usetex=True)

plt.rc('font', family='serif')

import pandas as pd
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"  
from dataset.utils.fungi_vis import FungiTasticVis

from types import SimpleNamespace
# import SimpleNamespace

#  fix random seeds for reproducibility
import random
import numpy as np
from PIL import Image, ImageOps
import tensorflow as tf
from pathlib import Path
from collections import Counter
import cv2
random.seed(0)
np.random.seed(0)

In [30]:
valset = FungiTasticVis(
        root=f"{os.getcwd()}/dataset/FungiTastic/",
        split='val',
        size='300',
        task='open',
        data_subset='Mini',
        transform=None,
)

trainset = FungiTasticVis(
        root=f"{os.getcwd()}/dataset/FungiTastic/",
        split='train',
        size='300',
        task='closed',
        data_subset='Mini',
        transform=None,
)

testset = FungiTasticVis(
        root=f"{os.getcwd()}/dataset/FungiTastic/",
        split='test',
        size='300',
        task='closed',
        data_subset='Mini',
        transform=None,
)

### Data preprocessing

In this code chunk, I am defining functions that first extract the training paths and labels from the datasets loaded above, and then define a function that creates a tf.dataset so that not all images need to be loaded into memory at once (will slow computer or crash).

In [119]:
from tqdm import tqdm

def extract_paths_and_labels(ds, expect_labels=True):
    paths, labels = [], []
    for i in tqdm(range(len(ds)), desc="Indexing"):
        _, y, p = ds[i]            # (PIL_image, label, path)
        paths.append(str(p))
        if expect_labels:
            if y is None:
                raise ValueError("Dataset has unlabeled samples but expect_labels=True.")
            labels.append(int(y))  # adjust if your labels are strings
    return (paths, labels) if expect_labels else (paths, None)

train_paths, train_labels = extract_paths_and_labels(trainset, expect_labels=True)
val_paths,   val_labels   = extract_paths_and_labels(valset,   expect_labels=True)
test_paths,  _            = extract_paths_and_labels(testset,  expect_labels=False)  # labels are None

Indexing: 100%|██████████| 10738/10738 [00:19<00:00, 564.86it/s]


In [ ]:
AUTOTUNE   = tf.data.AUTOTUNE
BATCH      = 64
HEIGHT, WIDTH = 224, 224

def decode_and_resize_from_path(path):
    bytes_ = tf.io.read_file(path)
    img = tf.image.decode_image(bytes_, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, [HEIGHT, WIDTH],
                          method=tf.image.ResizeMethod.LANCZOS5)  # or BICUBIC
    img = tf.clip_by_value(img, 0.0, 255.0)    # <-- keep within range
    img = tf.cast(img, tf.float32) / 255.0
    return img

In [69]:
def make_supervised_ds(paths, labels, training=True, shuffle_buf=10_000):
    """
    This function creates a tf dataset, which loads and preprocesses the data in batches of 64 images. It resizes them to be (224,224) images,
    randomly shuffles the data, and also randomly flips images horizontally.
    """
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(shuffle_buf, reshuffle_each_iteration=True)
    ds = ds.map(lambda p, y: (decode_and_resize_from_path(p), tf.cast(y, tf.int32)),
                num_parallel_calls=AUTOTUNE)
    ds = ds.ignore_errors()

    if training:
        ds = ds.map(lambda x, y: (tf.image.random_flip_left_right(x), y),
                    num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH, drop_remainder=training).prefetch(AUTOTUNE)

    opts = tf.data.Options(); opts.experimental_deterministic = False
    return ds.with_options(opts)

train_ds = make_supervised_ds(train_paths, train_labels, training=True)
val_ds   = make_supervised_ds(val_paths,   val_labels,   training=False)

In [64]:
#Sanity check
for images, labels in train_ds.take(1):
    print("Image batch shape:", images.shape)
    print("Label batch shape:", labels.shape)
    print("dtype:", images.dtype)
    print("Min pixel value:", tf.reduce_min(images).numpy())
    print("Max pixel value:", tf.reduce_max(images).numpy())

Image batch shape: (64, 224, 224, 3)
Label batch shape: (64,)
dtype: <dtype: 'float32'>
Min pixel value: 0.0
Max pixel value: 1.0


2025-11-09 16:40:46.016561: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


### Baseline Model: Majority Class Predictor

In [115]:
majority_label = Counter(train_labels).most_common(1)[0][0]
print("Majority class:", majority_label)

Majority class: 83


In [ ]:
def majority_accuracy(dataset, majority_label):
    total = 0
    correct = 0
    for i, (_, y) in enumerate(dataset):
        y_np = y.numpy()
        pred = np.full_like(y_np, fill_value=majority_label)
        correct += (pred == y_np).sum()
        total += y_np.size
    return correct / total

maj_train_acc = majority_accuracy(train_ds, majority_label)
maj_val_acc   = majority_accuracy(val_ds,   majority_label)
print(f"Majority baseline — train = {maj_train_acc:.3f}, val={maj_val_acc:.3f}")